# Diagnostic: HapMap3 from the rsID list to the QC'd panel, step by step

In [ ]:
RUN_ID = "sim"
FORCE = False

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = cfg.params.get("threads", 4)
if cfg.params.get("scratch"):   # mktemp in %%bash, tempfile, and R use TMPDIR
    import tempfile
    tempfile.tempdir = os.environ["TMPDIR"] = os.path.expanduser(cfg.params["scratch"])
    os.makedirs(tempfile.tempdir, exist_ok=True)

def setenv(**kw):
    """Variables for the %%bash / %%script cells below."""
    os.environ.update({k: str(v) for k, v in kw.items()})

def mark_done(stage, uri, **extra):
    """Provenance sidecar for an output a shell or R cell just wrote."""
    write_meta(get_store(uri), uri, collect_meta(cfg, stage, uri, extra=extra))

## Inputs

In [ ]:
STAGE = "hm3_overlap"
rep, OUT = Report(cfg, STAGE), single_target(cfg, STAGE)
WORK = tempfile.mkdtemp(prefix="hm3_overlap_")
geno_key = "acaf_bfile" if "acaf_bfile" in cfg.params["inputs"] else "acaf_pfile"
setenv(HM3=cfg.input("hm3_rsids"), REF_PVAR=cfg.input("ref_pvar_rsid"), WORK=WORK, NCHR=22,
       ACAF=cfg.input(geno_key, chr="{chr}") + (".bim" if geno_key == "acaf_bfile" else ".pvar"),
       BIM=int(geno_key == "acaf_bfile"), BILLING=cfg.params.get("billing_project", ""))

## rsIDs to GRCh38 positions (1000G)

In [ ]:
%%bash
set -euo pipefail
rd() {   # cat a local, gs:// or https:// file, decompressing .zst / .gz
  case "$1" in gs://*) gsutil -q ${BILLING:+-u "$BILLING"} cat "$1" ;; http*) curl -fsSL --retry 5 "$1" ;; *) cat "$1" ;; esac |
  case "${1%%\?*}" in *.zst) zstdcat ;; *.gz) gzip -dc ;; *) cat ;; esac
}
rd "$HM3" | awk '!(FNR == 1 && $1 == "SNP") {print $1}' | LC_ALL=C sort -u > "$WORK/hm3.rsids"
# chr pos rsid ref alt, for every 1000G record carrying an HM3 rsID
rd "$REF_PVAR" | awk -v OFS='\t' 'NR == FNR {h[$1]; next} !/^#/ && ($3 in h) {sub(/^chr/, "", $1); print $1, $2, $3, $4, $5}' \
  "$WORK/hm3.rsids" - > "$WORK/hm3_1kg.tsv"
wc -l "$WORK/hm3.rsids" "$WORK/hm3_1kg.tsv"

## The same positions in ACAF

In [ ]:
%%bash
set -euo pipefail
rd() {   # cat a local, gs:// or https:// file, decompressing .zst / .gz
  case "$1" in gs://*) gsutil -q ${BILLING:+-u "$BILLING"} cat "$1" ;; http*) curl -fsSL --retry 5 "$1" ;; *) cat "$1" ;; esac |
  case "${1%%\?*}" in *.zst) zstdcat ;; *.gz) gzip -dc ;; *) cat ;; esac
}
cut -f1,2 "$WORK/hm3_1kg.tsv" | LC_ALL=C sort -u > "$WORK/pos.tsv"
: > "$WORK/acaf_hits.tsv"
for c in $(seq 1 "$NCHR"); do
  f=${ACAF//\{chr\}/$c}
  if [ "$BIM" = 1 ]; then rd "$f" | awk -v OFS='\t' '{sub(/^chr/, "", $1); print $1, $4, $6, $5}'   # bim: A2 = REF
  else rd "$f" | awk -v OFS='\t' '!/^#/ {sub(/^chr/, "", $1); print $1, $2, $4, $5}'; fi |
    awk -v OFS='\t' 'NR == FNR {h[$1":"$2]; next} ($1":"$2) in h' "$WORK/pos.tsv" - >> "$WORK/acaf_hits.tsv"
done
wc -l "$WORK/acaf_hits.tsv"

## Counts

In [ ]:
names = ["chr", "pos", "ref", "alt"]
kg = pd.read_csv(f"{WORK}/hm3_1kg.tsv", sep="\t", header=None, names=["chr", "pos", "rsid", "ref", "alt"], dtype=str)
acaf = pd.read_csv(f"{WORK}/acaf_hits.tsv", sep="\t", header=None, names=names, dtype=str)
n_rsids = sum(1 for _ in open(f"{WORK}/hm3.rsids"))

key = lambda d, a="ref", b="alt": d["chr"] + ":" + d["pos"] + ":" + d[a] + ":" + d[b]
bi = kg[~kg["alt"].str.contains(",") & ~kg["rsid"].duplicated(keep=False)]
pos = bi["chr"] + ":" + bi["pos"]
acaf_pos = acaf["chr"] + ":" + acaf["pos"]
multi_pos = set(acaf_pos[acaf["alt"].str.contains(",") | acaf_pos.duplicated(keep=False)])
acaf_keys = set(key(acaf))
exact = key(bi).isin(acaf_keys)
swapped = ~exact & key(bi, "alt", "ref").isin(acaf_keys)
amb = np.array([(r, a) in {("A", "T"), ("T", "A"), ("C", "G"), ("G", "C")} for r, a in zip(bi["ref"], bi["alt"])])

steps = pd.DataFrame([
    ("HM3 rsIDs", n_rsids),
    ("found in 1000G GRCh38", kg["rsid"].nunique()),
    ("biallelic, unique in 1000G", len(bi)),
    ("position present in ACAF", int(pos.isin(set(acaf_pos)).sum())),
    ("ID+REF+ALT match in ACAF", int(exact.sum())),
    ("  of which multi-allelic site in ACAF", int((exact & pos.isin(multi_pos)).sum())),
    ("  of which A/T or C/G", int((exact & amb).sum())),
    ("REF/ALT swapped vs ACAF (lost)", int(swapped.sum())),
], columns=["step", "variants"])

panel_counts = single_target(cfg, "hm3_panel")[:-4] + ".counts.tsv"
if get_store(panel_counts).exists(panel_counts):
    pc = pd.read_csv(panel_counts, sep="\t").set_index("step")["variants"]
    steps = pd.concat([steps, pd.DataFrame([(f"hm3_panel: {k}", int(v)) for k, v in pc.items() if k != "samples"],
                                           columns=["step", "variants"])], ignore_index=True)
steps["pct_of_hm3"] = (100 * steps["variants"] / n_rsids).round(1)
rep.table(steps, "hm3_overlap")
steps

In [ ]:
fig, ax = plt.subplots(figsize=(8, 0.4 * len(steps) + 1))
ax.barh(steps["step"], steps["variants"], color=["0.6" if s.startswith("  ") else "tab:blue" for s in steps["step"]])
ax.invert_yaxis(); ax.set_xlabel("variants")
rep.fig(fig, "hm3_overlap"); plt.show()

## Write (the matched IDs, in the pipeline's chr:pos:ref:alt form, can be `inputs.hm3_ids`)

In [ ]:
ids_uri = os.path.join(os.path.dirname(OUT), "hm3_ids_matched.txt")
get_store(ids_uri).write_text(ids_uri, "\n".join(key(bi[exact])) + "\n")
with record(cfg, STAGE, OUT) as meta:
    get_store(OUT).write_text(OUT, steps.to_csv(sep="\t", index=False))
    meta["ids"] = ids_uri
rep.note(ids=ids_uri)
rep.save()